# 00 — Data Fetching

Single source of truth for the data layer. This notebook assembles every input the EDA and SCM notebooks consume and writes it to `../data/`. Re-run it to refresh — **only the missing or stale tickers are downloaded** (per-ticker parquet cache in `data/donors/`).

**Outputs written to `../data/`**

| File | Source | Contents |
|---|---|---|
| `brent_spot.csv` | EIA via sibling folder `Global port supply-chains/petroleum_data/Brent_Spot.csv` | Daily Brent spot, USD/bbl, 2000-01-04 → today |
| `daily_panel.csv` | EIA via sibling folder | 19-column daily energy panel (WTI, NG, refined products, stocks) |
| `gpr_daily.parquet` | Caldara & Iacoviello via sibling `gpr_daily.xls` | Daily Geopolitical Risk index (Federal Reserve Board) |
| `donors/{name}.parquet` (one per donor) | Yahoo Finance via yfinance | **33 non-energy macro-financial donors** — per-ticker cache for incremental updates |
| `donors.parquet` | Assembled from the per-ticker caches above | Combined panel that downstream notebooks read |
| `historical_events.csv` | Hand-curated from the WEF *Big Chart of Oil Prices* | Oil-market events 2000–2026, tagged by category for the EDA plot |
| `donor_audit.csv` | Hand-curated per-event audit | Each donor × event marked clean / mild / heavy treatment for SUTVA assessment |

**Expanded donor pool (33 series):** see [donor_catalog.md](../docs/donor_catalog.md) for the full table of selected donors, excluded candidates, and per-event treatment status. Original 12 donors plus Silver, Platinum, Palladium, Corn, Coffee, Sugar, Cotton, Live Cattle, Nikkei, EM equities, 7 FX (GBP, AUD, CHF, CNY, INR, KRW, ZAR — plus original EUR, MXN, JPY renamed), TLT, HYG.

In [1]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
DATA.mkdir(exist_ok=True)
SIBLING = ROOT.parent / 'Global port supply-chains'

print(f'Project root  : {ROOT}')
print(f'Data dir      : {DATA}')
print(f'Sibling data  : {SIBLING}  (exists={SIBLING.exists()})')

Project root  : c:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis
Data dir      : c:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\data
Sibling data  : c:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Global port supply-chains  (exists=True)


## 1. Brent spot + energy panel

Copied from the sibling project so this folder is self-contained going forward. `Brent_Spot.csv` is the EIA daily spot series (`RBRTEd`); `daily_panel.csv` is the wider energy panel used by downstream notebooks for energy-complex EDA.

In [2]:
panel_src = SIBLING / 'petroleum_data' / 'daily_panel.csv'
brent_src = SIBLING / 'petroleum_data' / 'Brent_Spot.csv'

panel = pd.read_csv(panel_src, parse_dates=['Date']).sort_values('Date')
brent = pd.read_csv(brent_src, parse_dates=['Date']).sort_values('Date')

panel.to_csv(DATA / 'daily_panel.csv', index=False)
brent.to_csv(DATA / 'brent_spot.csv', index=False)

print(f'Daily panel : {panel.shape}, {panel["Date"].min().date()} -> {panel["Date"].max().date()}')
print(f'Brent spot  : {brent.shape}, {brent["Date"].min().date()} -> {brent["Date"].max().date()}')
print(f'Brent NaN   : {brent["Price"].isna().sum()}')
brent.tail(3)

Daily panel : (6869, 20), 2000-01-04 -> 2026-05-01
Brent spot  : (6679, 2), 2000-01-04 -> 2026-04-27
Brent NaN   : 0


,Date,Price
6676,2026-04-23,113.25
6677,2026-04-24,111.86
6678,2026-04-27,113.89


## 2. Geopolitical Risk index (Caldara & Iacoviello, 2022)

Daily index; not a donor (it spikes endogenously to the very events we're studying) — used as an overlay in the EDA / gap plots only. Saving as parquet so subsequent notebooks don't need `xlrd`.

In [3]:
gpr_src = SIBLING / 'external_controls' / 'gpr_daily.xls'
gpr = pd.read_excel(gpr_src)
gpr['date'] = pd.to_datetime(gpr['DAY'].astype(str), format='%Y%m%d')
gpr = gpr.set_index('date')[['GPRD']].sort_index()
gpr.to_parquet(DATA / 'gpr_daily.parquet')

print(f'GPR daily : {gpr.shape}, {gpr.index.min().date()} -> {gpr.index.max().date()}')
gpr.tail(3).round(2)

GPR daily : (15096, 1), 1985-01-01 -> 2026-05-01


,GPRD
date,
2026-04-29,252.77
2026-04-30,207.04
2026-05-01,178.66


## 3. Macro-financial donors via yfinance

Expanded 33-series donor pool with **per-ticker parquet caching** in `data/donors/`. The `fetch_or_load` helper:

- Skips download if a cached parquet exists and was last updated less than `refresh_threshold_days` days ago.
- Otherwise fetches incrementally from `last cached date + 1` to today.
- Falls back to the cache if Yahoo returns an error (resilient to API hiccups).

To force a fresh download of one donor, delete its `data/donors/{name}.parquet` file. To add a new donor, append a row to `TICKERS` below — only that ticker will be fetched on the next run.

Selection rationale and excluded candidates (CAD, NOK, BRL, RUB, TRY petrocurrencies; WTI/NG/refined-products energy complex) are documented in [donor_catalog.md](../docs/donor_catalog.md).

In [4]:
import yfinance as yf

DONORS_DIR = DATA / 'donors'
DONORS_DIR.mkdir(exist_ok=True)

TICKERS = {
    # --- Metals (6): industrial cycle + precious + auto/Russia exposure ---
    'Copper':     'HG=F',
    'IronOre':    'TIO=F',
    'Silver':     'SI=F',
    'Platinum':   'PL=F',
    'Palladium':  'PA=F',
    'Gold':       'GC=F',

    # --- Agricultural (7) ---
    'Soybeans':   'ZS=F',
    'Wheat':      'ZW=F',
    'Corn':       'ZC=F',
    'Coffee':     'KC=F',
    'Sugar':      'SB=F',
    'Cotton':     'CT=F',
    'LiveCattle': 'LE=F',

    # --- Equities (4) ---
    'SP500':      'SPY',
    'WorldEq':    'URTH',
    'Nikkei':     '^N225',
    'EM_Eq':      'EEM',

    # --- FX (11): direction documented in donor_catalog.md ---
    'EUR':        'EURUSD=X',   # USD per 1 EUR  (up = EUR strong, USD weak)
    'GBP':        'GBPUSD=X',   # USD per 1 GBP
    'AUD':        'AUDUSD=X',   # USD per 1 AUD
    'JPY':        'JPY=X',      # JPY per 1 USD  (up = USD strong, JPY weak)
    'CHF':        'CHF=X',      # CHF per 1 USD
    'CNY':        'CNY=X',      # CNY per 1 USD
    'INR':        'INR=X',      # INR per 1 USD
    'KRW':        'KRW=X',      # KRW per 1 USD
    'ZAR':        'ZAR=X',      # ZAR per 1 USD
    'MXN':        'MXN=X',      # MXN per 1 USD
    'DXY':        'DX-Y.NYB',   # broad trade-weighted USD index

    # --- Rates & credit (3) ---
    'US10Y':      '^TNX',
    'TLT':        'TLT',
    'HYG':        'HYG',

    # --- Volatility (1) ---
    'VIX':        '^VIX',

    # --- Crypto (1) ---
    'BTC':        'BTC-USD',
}

assert len(TICKERS) == 33, f'expected 33 donors, got {len(TICKERS)}'


def fetch_or_load(name, ticker, refresh_threshold_days=3):
    """Load per-ticker parquet cache if recent; otherwise fetch (incrementally if possible).

    Returns: (Series, status_str, last_date_str)
    """
    cache_path = DONORS_DIR / f'{name}.parquet'
    today = pd.Timestamp.today().normalize()

    cached, last = None, None
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        cached = df[name] if name in df.columns else df.iloc[:, 0].rename(name)
        last = cached.dropna().index.max()
        if pd.notna(last) and (today - last).days <= refresh_threshold_days:
            return cached, 'cached', str(last.date())

    fetch_start = (last + pd.Timedelta(days=1)).strftime('%Y-%m-%d') if cached is not None and pd.notna(last) else '2010-01-01'
    end_date = (today + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    try:
        raw = yf.download(ticker, start=fetch_start, end=end_date,
                          progress=False, auto_adjust=False)
    except Exception as e:
        if cached is not None:
            return cached, f'fetch err ({str(e)[:25]})', str(last.date())
        return pd.Series(name=name, dtype=float), f'fetch err ({str(e)[:25]})', 'N/A'

    if len(raw) == 0:
        if cached is not None:
            return cached, 'no new data', str(last.date())
        return pd.Series(name=name, dtype=float), 'empty fetch', 'N/A'

    new = raw['Close']
    if isinstance(new, pd.DataFrame):
        new = new.iloc[:, 0]
    new.name = name
    new.index.name = 'date'

    if cached is not None:
        combined = pd.concat([cached, new])
        combined = combined[~combined.index.duplicated(keep='last')].sort_index()
        status = f'updated (+{len(new)})'
    else:
        combined = new.sort_index()
        status = 'fresh fetch'

    combined.to_frame().to_parquet(cache_path)
    return combined, status, str(combined.dropna().index.max().date())


print(f'Per-ticker cache dir : {DONORS_DIR}')
print(f'Existing cache files : {len(list(DONORS_DIR.glob("*.parquet")))}')
print()

donor_series = {}
for name, ticker in TICKERS.items():
    s, status, last = fetch_or_load(name, ticker)
    donor_series[name] = s
    print(f'  {name:11s} ({ticker:10s})  {status:25s}  -> last {last}')

donors = pd.DataFrame(donor_series)[list(TICKERS.keys())]
donors.index.name = 'date'
donors.to_parquet(DATA / 'donors.parquet')

print()
print(f'Combined panel : {donors.shape}, total donors = {donors.shape[1]}')
print(f'\nFirst non-NaN per donor:')
print(donors.apply(lambda s: s.dropna().index.min().date() if s.notna().any() else None).to_string())

Per-ticker cache dir : c:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\data\donors
Existing cache files : 33

  Copper      (HG=F      )  updated (+3)               -> last 2026-05-26
  IronOre     (TIO=F     )  updated (+3)               -> last 2026-05-22
  Silver      (SI=F      )  updated (+3)               -> last 2026-05-26
  Platinum    (PL=F      )  updated (+3)               -> last 2026-05-26
  Palladium   (PA=F      )  updated (+3)               -> last 2026-05-26
  Gold        (GC=F      )  updated (+3)               -> last 2026-05-26
  Soybeans    (ZS=F      )  updated (+3)               -> last 2026-05-26
  Wheat       (ZW=F      )  updated (+3)               -> last 2026-05-26
  Corn        (ZC=F      )  updated (+3)               -> last 2026-05-26
  Coffee      (KC=F      )  updated (+3)               -> last 2026-05-26
  Sugar       (SB=F      )  updated (+3)               -> last 2026-05-26
  Cotton      (CT=F      )  updated (+4)               -> last 2

## 4. Historical events table

Hand-curated from the WEF *Big Chart of Oil Prices*. **Categories** are the key analytical dimension for the donor-independence test in the EDA:

| Category | Meaning | What we expect the donors to do |
|---|---|---|
| `supply` | Chokepoint / OPEC-driven oil-supply shock | Donors should **not** move much — this is the gap SCM is built to detect |
| `policy` | Sanctions / production-policy decisions | Donors should not move much — same logic |
| `geopolitics` | MENA/Russia geopolitical shock | Donors may pick up some risk-premium component via VIX/Gold/equities |
| `demand` | Global macro/financial shock | Donors **should** move (this is the co-movement SCM exploits to build the counterfactual) |

If donors track Brent on `supply`/`policy` events, the donor pool is contaminated. If they track Brent only on `demand`/`geopolitics` events, the SCM identifying assumption is supported.

In [5]:
events = pd.DataFrame([
    ('2001-09-11', '9/11',                 'demand',      'Terrorist attacks; global risk-off, demand collapse'),
    ('2003-03-20', 'Iraq War',              'geopolitics', 'US-led invasion of Iraq; Middle East supply premium'),
    ('2005-08-29', 'Hurricane Katrina',     'supply',      'Gulf-of-Mexico production and refining outages'),
    ('2008-07-11', 'Brent peak $147',       'demand',      'Pre-crisis demand peak; speculative top'),
    ('2008-09-15', 'Lehman / GFC',          'demand',      'Global financial crisis; demand and risk collapse'),
    ('2010-12-17', 'Arab Spring begins',    'geopolitics', 'Tunisian self-immolation; MENA political unrest cascade'),
    ('2011-02-15', 'Libyan civil war',      'supply',      'Libyan crude offline; ~1.6 mb/d disruption'),
    ('2012-07-01', 'EU Iran embargo',       'policy',      'EU embargo on Iranian crude imports takes effect'),
    ('2014-11-27', 'OPEC declines to cut',  'policy',      'Saudi Arabia signals defence of market share; oil glut begins'),
    ('2016-02-11', 'Brent trough ~$28',     'policy',      'Bottom of the OPEC price war / shale-glut cycle'),
    ('2016-11-30', 'OPEC+ deal',            'policy',      'First OPEC+ production-cut agreement with Russia'),
    ('2018-05-08', 'US exits JCPOA',        'policy',      'Iran sanctions reimposed; export reductions begin'),
    ('2019-09-14', 'Abqaiq attack',         'supply',      'Drone strike on Saudi Abqaiq facility; 5.7 mb/d offline briefly'),
    ('2020-03-08', 'Saudi-Russia price war','policy',      'OPEC+ collapses; Saudi raises output, prices crash'),
    ('2020-03-11', 'COVID-19 pandemic',     'demand',      'WHO declares pandemic; global demand collapse'),
    ('2020-04-20', 'WTI negative',          'supply',      'Cushing storage saturates; WTI front-month settles -$37 (Brent stayed positive)'),
    ('2021-10-01', 'Energy-cycle reflation','demand',      'Post-COVID demand recovery; commodity supercycle narrative'),
    ('2022-02-24', 'Russia invades Ukraine','geopolitics', 'War; Russian oil sanctions; Brent spikes to ~$130'),
    ('2022-12-05', 'Russia oil price cap',  'policy',      'G7+EU $60/bbl price cap on Russian seaborne crude'),
    ('2023-10-07', 'Israel-Hamas war',      'geopolitics', 'Hamas attack on Israel; MENA risk premium reactivated'),
    ('2023-11-19', 'Red Sea diversion',     'supply',      'Galaxy Leader hijack; Bab-el-Mandeb container diversion begins'),
    ('2026-02-01', 'Hormuz crisis',         'supply',      'Strait of Hormuz disruption; main thesis subject'),
], columns=['date', 'label', 'category', 'description'])

events['date'] = pd.to_datetime(events['date'])
events = events.sort_values('date').reset_index(drop=True)
events.to_csv(DATA / 'historical_events.csv', index=False)

print(f'Events : {len(events)}')
print(events['category'].value_counts().to_string())
events

Events : 22
category
policy         7
supply         6
demand         5
geopolitics    4


,date,label,category,description
0,2001-09-11,9/11,demand,"Terrorist attacks; global risk-off, demand col..."
1,2003-03-20,Iraq War,geopolitics,US-led invasion of Iraq; Middle East supply pr...
2,2005-08-29,Hurricane Katrina,supply,Gulf-of-Mexico production and refining outages
3,2008-07-11,Brent peak $147,demand,Pre-crisis demand peak; speculative top
4,2008-09-15,Lehman / GFC,demand,Global financial crisis; demand and risk collapse
5,2010-12-17,Arab Spring begins,geopolitics,Tunisian self-immolation; MENA political unres...
6,2011-02-15,Libyan civil war,supply,Libyan crude offline; ~1.6 mb/d disruption
7,2012-07-01,EU Iran embargo,policy,EU embargo on Iranian crude imports takes effect
8,2014-11-27,OPEC declines to cut,policy,Saudi Arabia signals defence of market share; ...
9,2016-02-11,Brent trough ~$28,policy,Bottom of the OPEC price war / shale-glut cycle


## 4b. Per-event donor audit (SUTVA assessment)

Two focal events: **Russia invades Ukraine (2022-02-24)** and **Strait of Hormuz crisis (2026-02-01)** — both selected because Brent visibly moved, so there is an actual magnitude to estimate and validate. Red Sea 2023 is *not* a focal event (no visible Brent disruption — see [donor_catalog.md](../docs/donor_catalog.md) for the reasoning).

For each focal event we mark every donor as:

- **C** — *clean*: not affected by the event through any direct supply/demand channel; co-movement with Brent is via the intended common-factor channels (growth, dollar, risk premium).
- **M** — *mildly treated*: secondary or cross-elasticity exposure that may bias the estimate slightly.
- **H** — *heavily treated*: direct supply/demand exposure to the event itself; **must be excluded** from the SCM donor pool for that event.

The SCM notebook uses this table to construct a per-event clean donor subset. Hormuz is clean across the pool (no Persian Gulf routing for any donor); Russia 2022 is where the donor exclusions bite (6 heavy + 6 mild = 12 donors excluded from the strict-clean 21-donor pool).

In [6]:
# Per-event donor audit. Status codes:  C = clean,  M = mildly treated,  H = heavily treated.
# Reasoning embedded in the 'reason' column where it isn't obvious.
# For full justification of each row, see donor_catalog.md.
#
# Focal events: Russia 2022 + Hormuz 2026 (both produced visible Brent disruption).
# Red Sea 2023 is intentionally NOT a focal event — no visible Brent disruption,
# only a *null* validation case (weaker than magnitude validation from Russia 2022).

DONOR_AUDIT_RAW = [
    # donor       Russia_2022  Hormuz_2026   reason for any non-C entry
    # --- Metals ---
    ('Copper',    'M',         'C',          'Russia ~4% world copper supply'),
    ('IronOre',   'M',         'C',          'Russia ~5% world iron ore exports'),
    ('Silver',    'C',         'C',          ''),
    ('Platinum',  'C',         'C',          'S.Africa-dominated; no Russia exposure'),
    ('Palladium', 'H',         'C',          'Russia ~40% world supply; price spiked 2022'),
    ('Gold',      'C',         'C',          'safe-haven channel = intended SCM co-movement'),
    # --- Agricultural ---
    ('Soybeans',  'M',         'C',          'cross-elasticity with wheat / Ukrainian sunflower oil'),
    ('Wheat',     'H',         'C',          'Russia+Ukraine ~30% world exports; ZW=F +80% Mar2022'),
    ('Corn',      'H',         'C',          'Ukraine ~13% world corn exports'),
    ('Coffee',    'C',         'C',          'Brazil/Vietnam concentrated'),
    ('Sugar',     'C',         'C',          'Brazil/India; clean'),
    ('Cotton',    'C',         'C',          'US/China/India; clean'),
    ('LiveCattle','C',         'C',          'N.America domestic; clean'),
    # --- Equities ---
    ('SP500',     'C',         'C',          'risk-premium channel = intended SCM co-movement'),
    ('WorldEq',   'C',         'C',          'same'),
    ('Nikkei',    'C',         'C',          'Japan equity; clean'),
    ('EM_Eq',     'M',         'C',          'broad EM ETF; some Russia weight pre-exclusion'),
    # --- FX ---
    ('EUR',       'H',         'C',          'European energy-import crisis hit EUR specifically'),
    ('GBP',       'M',         'C',          'UK gas-market exposure to Russia crisis'),
    ('AUD',       'C',         'C',          'Australia commodity exporter; clean of Russia'),
    ('JPY',       'C',         'C',          'safe-haven FX; clean'),
    ('CHF',       'C',         'C',          'Swiss neutrality + safe-haven; clean'),
    ('CNY',       'C',         'C',          'China managed currency; stayed out of Russia sanctions'),
    ('INR',       'C',         'C',          'India EM FX; no direct Russia channel'),
    ('KRW',       'C',         'C',          'Korea manufacturing FX; clean'),
    ('ZAR',       'C',         'C',          'S.Africa EM FX; clean'),
    ('MXN',       'C',         'C',          'Mexico EM FX; clean of Russia'),
    ('DXY',       'H',         'C',          'dollar appreciation on European crisis = treated via EUR weight in basket'),
    # --- Rates & credit ---
    ('US10Y',     'M',         'C',          'inflation-expectations channel via European HICP'),
    ('TLT',       'C',         'C',          'long-duration US Treasuries; clean'),
    ('HYG',       'C',         'C',          'US high-yield credit; clean'),
    # --- Vol & crypto ---
    ('VIX',       'C',         'C',          'risk-premium channel = intended SCM co-movement'),
    ('BTC',       'H',         'C',          'sanctions-evasion narrative drove abnormal flow Mar2022'),
]

audit = pd.DataFrame(DONOR_AUDIT_RAW,
                     columns=['donor', 'Russia_2022', 'Hormuz_2026', 'reason'])
assert len(audit) == 33, f'expected 33 audit rows, got {len(audit)}'
audit.to_csv(DATA / 'donor_audit.csv', index=False)

print(f'Donor audit : {len(audit)} donors x 2 focal events')
print('\nCount by status:')
for ev in ['Russia_2022', 'Hormuz_2026']:
    counts = audit[ev].value_counts().to_dict()
    print(f'  {ev:14s}  C={counts.get("C", 0)}  M={counts.get("M", 0)}  H={counts.get("H", 0)}')
print()
print('Russia 2022 — heavily treated (excluded from clean pool):',
      audit.loc[audit['Russia_2022'] == 'H', 'donor'].tolist())
print('Russia 2022 — mildly treated:',
      audit.loc[audit['Russia_2022'] == 'M', 'donor'].tolist())
audit

Donor audit : 33 donors x 2 focal events

Count by status:
  Russia_2022     C=21  M=6  H=6
  Hormuz_2026     C=33  M=0  H=0

Russia 2022 — heavily treated (excluded from clean pool): ['Palladium', 'Wheat', 'Corn', 'EUR', 'DXY', 'BTC']
Russia 2022 — mildly treated: ['Copper', 'IronOre', 'Soybeans', 'EM_Eq', 'GBP', 'US10Y']


,donor,Russia_2022,Hormuz_2026,reason
0,Copper,M,C,Russia ~4% world copper supply
1,IronOre,M,C,Russia ~5% world iron ore exports
2,Silver,C,C,
3,Platinum,C,C,S.Africa-dominated; no Russia exposure
4,Palladium,H,C,Russia ~40% world supply; price spiked 2022
5,Gold,C,C,safe-haven channel = intended SCM co-movement
6,Soybeans,M,C,cross-elasticity with wheat / Ukrainian sunflo...
7,Wheat,H,C,Russia+Ukraine ~30% world exports; ZW=F +80% M...
8,Corn,H,C,Ukraine ~13% world corn exports
9,Coffee,C,C,Brazil/Vietnam concentrated


## 5. Verify data directory

In [7]:
print(f'Files in {DATA}:\n')
for f in sorted(DATA.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:30s}  {size_kb:>10.1f} KB')

Files in c:\EC\BSE\DSDM\Term 3\21DM000 Master Project\Brent Analysis\data:

  brent_spot.csv                       117.9 KB
  daily_panel.csv                     1472.5 KB
  donor_audit.csv                        1.6 KB
  donors                                12.0 KB
  donors.parquet                       816.1 KB
  gpr_daily.parquet                    211.1 KB
  historical_events.csv                  2.0 KB
  results                                0.0 KB
  validation                            28.0 KB
